# DADA3: 0 - Start here

No TODOs in this notebook.

In this assignment you will build a team of agents that works through the open issues of a small Python repository, the way a maintainer does: decide which issues are real bugs, fix the bugs, check each fix by running it, and write a report. `DADA3_SPEC.md` describes the whole assignment. Read it first.

This notebook sets up your environment, shows you the repository and its issues, and runs the repository's tests in a sandbox by hand, so that you know what the agents will be working with.

![architecture](figures/architecture.png)

## Setup

1. From this folder, in the VS Code terminal, install the packages: `uv pip install -r requirements.txt`. This includes `deepagents` and `langchain-daytona` from Week 9.
2. Copy `example.env` to `.env` and fill in two keys: `GOOGLE_API_KEY`, as in every assignment, and `DAYTONA_API_KEY`, the key you made for W09b.2.
3. If you would rather run the sandboxes in Docker on your own machine, as in section 2 of W09b.2, add `SANDBOX=docker` to `.env`, start Docker Desktop, and run `docker pull python:3.12-slim` once.

On a Mac with Python from python.org, the first call to Daytona can fail with `CERTIFICATE_VERIFY_FAILED`. `bugteam/sandbox.py` points Python to the `certifi` certificates, as W09b.2 did, so you should not see it. If you do, run `"/Applications/Python 3.13/Install Certificates.command"` in a terminal, with your Python version, and restart the kernel.

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
from bugteam import config

print("chat model:", config.CHAT_BACKEND, config.model_name())
print("sandbox:   ", config.SANDBOX)
assert config.CHAT_BACKEND != "gemini" or os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY in .env"
assert config.SANDBOX != "daytona" or os.getenv("DAYTONA_API_KEY"), "No DAYTONA_API_KEY in .env"

chat model: gemini gemini-3.5-flash-lite
sandbox:    daytona


## 1. The repository

`huskyutils` is a small library of helper functions that the Husky Tech web team uses in its store: shipping prices, splitting a payment between friends, delivery dates, product-card text, and a few text and list utilities. It is in the `repo/` folder of this bundle.

Every function has a docstring, and the repository's README says that each docstring is the function's specification: if the code and the docstring disagree, the code is wrong. Keep that rule in mind. Your agents will need it to tell a bug from a misunderstanding.

`repo.repo_files()` returns every file as a dictionary from its path to its text. That dictionary is how the agents read the repository.

In [2]:
from bugteam import repo

files = repo.repo_files()
for path in files:
    print(path)

print()
print(files["huskyutils/money.py"])

README.md
huskyutils/__init__.py
huskyutils/cards.py
huskyutils/dates.py
huskyutils/intervals.py
huskyutils/money.py
huskyutils/shipping.py
huskyutils/stats.py
huskyutils/text.py
tests/test_cards.py
tests/test_dates.py
tests/test_intervals.py
tests/test_money.py
tests/test_shipping.py
tests/test_stats.py
tests/test_text.py

"""Payments."""


def split_amount(cents, people):
    """Split an amount in cents into `people` shares, for a shared order.

    The shares add up to exactly `cents` and differ by at most one cent. When the amount does not
    divide evenly, the first shares get the extra cent. A negative amount (a refund) is split the
    same way as the positive amount, with every share negative. `people` is at least 1.

        split_amount(900, 3) == [300, 300, 300]
        split_amount(1000, 3) == [334, 333, 333]
        split_amount(-1000, 3) == [-334, -333, -333]
    """
    share = cents // people
    return [share] * people



## 2. The issues

The repository has ten open issues, written by people on the Husky Tech team who use the library. `issues.json` holds them in the shape GitHub's API returns them.

Read all ten before you go on. For each one, open the file it is about, read the docstring, and decide for yourself whether it is a bug, not a bug (the code does what its docstring says, and the reporter expected something else), or a request for something new. Write your ten labels down. In notebook 02 you will compare them with what your triage agent decides.

In [3]:
issues = repo.load_issues()
for issue in issues:
    print(repo.issue_text(issue))
    print("\n" + "-" * 80 + "\n")

Issue #1: Slugs have double hyphens and a hyphen at the end
(opened by store-frontend)

The product page for "Hello, World! T-shirt" got the URL `/p/hello--world--t-shirt`, and "Webcam (HD)" got `/p/webcam--hd-`.

```python
>>> from huskyutils.text import slugify
>>> slugify("Webcam (HD)")
'webcam--hd-'
```

I expected `webcam-hd`.

--------------------------------------------------------------------------------

Issue #2: Shared payment loses a cent
(opened by checkout-team)

Three friends split a $10.00 order and the charges were $3.33 each. That is $9.99, so one cent was never charged.

```python
>>> split_amount(1000, 3)
[333, 333, 333]
```

Suggested fix: add the leftover cents to the last person's share, which is how our point-of-sale system does it.

--------------------------------------------------------------------------------

Issue #3: word_count says "don't stop" is 2 words
(opened by reviews-moderator)

"don't" is short for "do not", so "don't stop" should be 3 words. Our

## 3. Running the tests in a sandbox

The repository has tests in `repo/tests/`, written with Python's `unittest`. We will run them in a sandbox, not on your machine. The tests here are harmless, but later in the assignment the code that runs is code an agent wrote, and W09b.2 showed why that belongs in a sandbox.

`bugteam/sandbox.py` holds the sandbox code from W09b.2. Read it now; it is short.

- `open_sandbox()` creates a sandbox with no network and deletes it when the `with` block ends, even if the code inside fails. It gives you the sandbox and `workdir`, the folder you can write to: `/home/daytona` on Daytona, `/workspace` in Docker.
- `upload_files(sandbox, files, folder)` copies a `{path: text}` dictionary into the sandbox.
- `read_sandbox_file(sandbox, path)` and `save_files(sandbox, folder, local_dir)` copy files back out.

In [4]:
from bugteam.sandbox import open_sandbox, upload_files

with open_sandbox() as (sandbox, workdir):
    upload_files(sandbox, files, f"{workdir}/repo")
    result = sandbox.execute(f"cd {workdir}/repo && python -m unittest discover -s tests -v")
    print(result.output)
    print("exit code:", result.exit_code)


<stderr>test_exact_length_unchanged (test_cards.TestTruncate.test_exact_length_unchanged) ... ok
test_long_text_ends_with_ellipsis (test_cards.TestTruncate.test_long_text_ends_with_ellipsis) ... ok
test_short_text_unchanged (test_cards.TestTruncate.test_short_text_unchanged) ... ok
test_friday_plus_one (test_dates.TestAddBusinessDays.test_friday_plus_one) ... ok
test_holiday_on_monday (test_dates.TestAddBusinessDays.test_holiday_on_monday) ... ok
test_within_a_week (test_dates.TestAddBusinessDays.test_within_a_week) ... ok
test_zero_days (test_dates.TestAddBusinessDays.test_zero_days) ... ok
test_does_not_modify_input (test_intervals.TestMergeIntervals.test_does_not_modify_input) ... ok
test_empty (test_intervals.TestMergeIntervals.test_empty) ... ok
test_overlapping (test_intervals.TestMergeIntervals.test_overlapping) ... ok
test_touching (test_intervals.TestMergeIntervals.test_touching) ... ok
test_even_refund (test_money.TestSplitAmount.test_even_refund) ... ok
test_even_split (tes

Every test passes, and yet the repository has seven real bugs among its issues. The tests check the cases someone thought of when the code was written, and the bugs are in the cases nobody tested. This is the ordinary state of a repository, and it is why a fix has to come with a new test: the old tests passing says nothing about the issue.

Here is one issue reproduced by hand, in a new sandbox. This is what the verifier agent in notebook 01 will do for every fix, with a test file instead of a one-line command.

In [5]:
with open_sandbox() as (sandbox, workdir):
    upload_files(sandbox, files, f"{workdir}/repo")
    command = 'python -c "from huskyutils.money import split_amount; print(split_amount(1000, 3))"'
    print(sandbox.execute(f"cd {workdir}/repo && {command}").output)

[333, 333, 333]



## 4. The model

Every agent in the team uses `config.chat()`. Check that it answers.

Mind the free-tier limits: 15 requests a minute and 500 a day. A team run on all ten issues is roughly 50 to 120 chat calls, depending on how many fixes the verifier sends back, and the deep agent's long system prompt makes every verifier call a large one. Develop with `CHAT_BACKEND=ollama` in `.env` if you can, so that building costs nothing, and save the Gemini quota for notebooks 02 and 03. Small local models handle the fixer well and the verifier poorly, so expect the verifier to stop without deciding when you develop with one.

In [6]:
from bugteam.util import retry, text_of

print(text_of(retry(config.chat().invoke, "In one sentence, what is a unit test?")))

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


A unit test is an automated piece of code written by a developer to verify that a small, isolated section of an application—typically a single function or method—behaves as expected.


## What comes next

- **Notebook 01** builds the fixer, the verifier, and the handoff between them, and runs the pair on one issue at a time.
- **Notebook 02** adds triage, runs one pair per bug in parallel, and writes the report.
- **Notebook 03** compares the team with a single deep agent doing the whole job, using tests the team never sees.